# 03 실전 반도체 공정 데이터 분석 · 3~4. 결측값 처리 · 분산 0 제거

- 강의 페이지: `Web/강좌/03_실전_반도체_공정_데이터분석/실전_반도체_공정_데이터분석_강의자료.html` → 목차 **결측값 처리 · 분산 0 제거**
- 결측이 많은 열을 지우고 중앙값으로 채운 뒤, 값이 변하지 않는 센서를 제거합니다.
- `fab.csv`가 이 노트북과 같은 폴더에 있어야 합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

### 3단계 · 결측값 처리 (적극 모드)
- 설명: 결측이 너무 많은 컬럼은 통째로 제거

### 준비 · 라이브러리 + 데이터 읽기 (1~2단계)

앞 단계 코드를 그대로 모아 한 번에 실행합니다. 출력은 앞 노트북과 같습니다.

In [ ]:
# ── 1단계 ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif   # ⭐ NEW
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# 그래프 한글 설정 (Mac은 'AppleGothic', Colab·리눅스는 'NanumGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 2단계 ──
df = pd.read_csv('fab.csv')


In [ ]:
# 1️⃣ 컬럼별 결측률 계산
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head(10))

In [ ]:
# 2️⃣ 결측률 50% 초과 컬럼 제거
THRESHOLD = 50.0
cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
print(f"🗑️ 제거할 컬럼: {len(cols_to_drop)}개")
df = df.drop(columns=cols_to_drop)

In [ ]:
# 3️⃣ 남은 결측치는 중앙값으로 채우기 (타겟 제외!)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')   # ⚠️ 타겟은 절대 채우지 말 것!

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
print(f"✅ 결측 처리 완료. 남은 결측: {df.isnull().sum().sum()}")

## 4단계 · 의미 없는 컬럼 제거 NEW

### 4단계 · 의미 없는 컬럼 제거 NEW
- 설명: 분산이 거의 없는 컬럼을 제거해 학습에 불필요한 변수를 줄입니다.
- 수업 메모: 분산이 0인 센서는 학습에 전혀 도움 안 됩니다

In [ ]:
# 분산 = 0 컬럼 + 거의 0인 컬럼 모두 제거
variances = df[numeric_cols].var()

constant_cols    = variances[variances == 0].index.tolist()
near_constant    = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

print(f"분산 0  컬럼: {len(constant_cols)}개")
print(f"분산 ≈0 컬럼: {len(near_constant)}개")

In [ ]:
to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]

print(f"✅ 사용 가능한 센서: {len(numeric_cols)}개")

## 마무리

- 결측률 50% 초과 28개 열 제거 → 중앙값 채우기 → 분산 0·≈0 센서 126개 제거로 **436개 센서**가 남습니다.